Transformer Encoder Layer 구현

In [1]:
import torch
import torch.nn as nn
import numpy as np
weights = {
    "W_q": np.random.randn(256, 256),  # Query projection weights (d_model, d_model)
    "W_k": np.random.randn(256, 256),  # Key projection weights   (d_model, d_model)
    "W_v": np.random.randn(256, 256),  # Value projection weights (d_model, d_model)
    "W_o": np.random.randn(256, 256),  # Output projection weights (d_model, d_model)
    "W_ff1": np.random.randn(512, 256),  # First Feed-Forward weights (d_ffn, d_model)
    "W_ff2": np.random.randn(256, 512),  # Second Feed-Forward weights (d_model, d_ffn)
    "gamma1": np.ones(256),  # LayerNorm 1 scale (gamma)    (d_model)
    "beta1": np.zeros(256),  # LayerNorm 1 shift (beta)     (d_model)
    "gamma2": np.ones(256),  # LayerNorm 2 scale (gamma)    (d_model)
    "beta2": np.zeros(256),  # LayerNorm 2 shift (beta)     (d_model)
}

"model의 d_model = 256, d_ffn=512, num_heads=8"

'model의 d_model = 256, d_ffn=512, num_heads=8'

In [2]:
import torch
import torch.nn as nn
import numpy as np

class MultiHeadAttentionTransformer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff):
        super(MultiHeadAttentionTransformer, self).__init__()
        self.self_attn = nn.MultiheadAttention(embed_dim=d_model, num_heads=num_heads, batch_first=True)
        self.W_ff1 = nn.Linear(d_model, d_ff, bias=False)
        self.W_ff2 = nn.Linear(d_ff, d_model, bias=False)
        self.norm1 = nn.LayerNorm(d_model, eps=1e-5)
        self.norm2 = nn.LayerNorm(d_model, eps=1e-5)

    def forward(self, x):
        # Step 1: Multi-Head Attention
        attn_output, _ = self.self_attn(x, x, x)
        attn_residual = self.norm1(x + attn_output)

        # Step 2: Feed-Forward Network
        ff_output = self.W_ff2(torch.relu(self.W_ff1(attn_residual)))
        ff_residual = self.norm2(attn_residual + ff_output)

        return ff_residual

# PyTorch 모델 초기화
pytorch_model_mha = MultiHeadAttentionTransformer(d_model=256, num_heads=8, d_ff=512)

# 가중치 동기화
with torch.no_grad():
    # Self-Attention weights
    W_qkv = torch.cat([torch.tensor(weights["W_q"], dtype=torch.float32),
                       torch.tensor(weights["W_k"], dtype=torch.float32),
                       torch.tensor(weights["W_v"], dtype=torch.float32)], dim=0)
    pytorch_model_mha.self_attn.in_proj_weight.copy_(W_qkv)
    pytorch_model_mha.self_attn.out_proj.weight.copy_(torch.tensor(weights["W_o"], dtype=torch.float32))
    pytorch_model_mha.self_attn.in_proj_bias.zero_()
    pytorch_model_mha.self_attn.out_proj.bias.zero_()

    # Feed-Forward weights
    pytorch_model_mha.W_ff1.weight.copy_(torch.tensor(weights["W_ff1"], dtype=torch.float32))
    pytorch_model_mha.W_ff2.weight.copy_(torch.tensor(weights["W_ff2"], dtype=torch.float32))

    # LayerNorm weights
    pytorch_model_mha.norm1.weight.copy_(torch.tensor(weights["gamma1"], dtype=torch.float32))
    pytorch_model_mha.norm1.bias.copy_(torch.tensor(weights["beta1"], dtype=torch.float32))
    pytorch_model_mha.norm2.weight.copy_(torch.tensor(weights["gamma2"], dtype=torch.float32))
    pytorch_model_mha.norm2.bias.copy_(torch.tensor(weights["beta2"], dtype=torch.float32))



In [3]:
import numpy as np

def softmax(x):
    e_x = np.exp(x - np.max(x, axis=-1, keepdims=True))  # 안정적인 계산을 위해 최대값 뺌
    return e_x / e_x.sum(axis=-1, keepdims=True)

def layer_norm(x, gamma, beta, eps=1e-5):
    mean = np.mean(x, axis=-1, keepdims=True)
    variance = np.var(x, axis=-1, keepdims=True)
    normalized_x = (x - mean) / np.sqrt(variance + eps)
    return gamma * normalized_x + beta


def multi_head_attention(Q, K, V, num_heads):
    """
    Multi-Head Self-Attention 수행
    """
    batch_size, seq_len, d_model = Q.shape
    head_dim = d_model // num_heads

    # Q, K, V reshape: (batch_size, seq_len, d_model) -> (batch_size, num_heads, seq_len, head_dim)
    Q = Q.reshape(batch_size, seq_len, num_heads, head_dim).transpose(0, 2, 1, 3)
    K = K.reshape(batch_size, seq_len, num_heads, head_dim).transpose(0, 2, 1, 3)
    V = V.reshape(batch_size, seq_len, num_heads, head_dim).transpose(0, 2, 1, 3)

    # Attention scores: (batch_size, num_heads, seq_len, seq_len)
    scores = np.matmul(Q, K.transpose(0, 1, 3, 2)) / np.sqrt(head_dim)
    attn_weights = softmax(scores)

    # Attention output: (batch_size, num_heads, seq_len, head_dim)
    attn_output = np.matmul(attn_weights, V)

    # Combine heads: (batch_size, num_heads, seq_len, head_dim) -> (batch_size, seq_len, d_model)
    attn_output = attn_output.transpose(0, 2, 1, 3).reshape(batch_size, seq_len, d_model)

    return attn_output

def python_transformer_block(input_data, weights, num_heads):
    # 가중치 분리
    W_q, W_k, W_v = weights["W_q"], weights["W_k"], weights["W_v"]
    W_o = weights["W_o"]
    W_ff1, W_ff2 = weights["W_ff1"], weights["W_ff2"]
    gamma1, beta1 = weights["gamma1"], weights["beta1"]
    gamma2, beta2 = weights["gamma2"], weights["beta2"]

    # Step 1: Multi-Head Self-Attention
    Q = np.dot(input_data, W_q.T)
    K = np.dot(input_data, W_k.T)
    V = np.dot(input_data, W_v.T)

    attn_output1 = multi_head_attention(Q, K, V, num_heads)
    attn_output2 = np.dot(attn_output1, W_o.T)  # Output projection

    # Residual connection + LayerNorm 1
    x = layer_norm(input_data + attn_output2, gamma1, beta1)

    # Step 2: Feed-Forward Network
    ff_output = np.dot(np.maximum(0, np.dot(x, W_ff1.T)), W_ff2.T)  # ReLU activation

    # Residual connection + LayerNorm 2
    output = layer_norm(x + ff_output, gamma2, beta2)
    return output




In [4]:
# 데이터 준비
input_data = np.random.randn(1, 50, 256).astype(np.float32)  # Batch size 1, seq_len 50, d_model 256

# Python 연산 수행
python_output = python_transformer_block(input_data, weights, num_heads=8)


# 입력 데이터
pytorch_input_mha = torch.tensor(input_data, dtype=torch.float32)

# PyTorch 모델 실행
pytorch_output_mha = pytorch_model_mha(pytorch_input_mha).detach().numpy()

for i in range(10):
    print(python_output[0][0][i], pytorch_output_mha[0][0][i])
print("Difference:", np.abs(python_output - pytorch_output_mha).mean())


0.5523063016075891 0.5523051
0.004633733964880846 0.0046366593
0.2355848576786642 0.23558415
-0.7064037581404065 -0.7064001
-0.2595307687859373 -0.25953242
0.47248138051910155 0.47247782
1.2619574652143466 1.2619541
-0.6831961810270446 -0.68319535
-0.5253113366805544 -0.5253102
0.8644963320016411 0.8644945
Difference: 1.3680547391954807e-06
